In [ ]:
import pyspark
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
spark = SparkSession.builder.master("local[*]").appName("lab_spark").getOrCreate()

In [ ]:
finance_info= spark.read.option("header",True).option("inferSchema",True).csv("fin_sample.csv")
#finance_info.printSchema()
df0=F.floor((F.to_timestamp(F.col("MOMENT")).cast('double')*1000).cast('long')/candle_width)
finance_info=finance_info.withColumn("candle",(df0))

In [ ]:
df=finance_info.select("#SYMBOL","MOMENT","PRICE_DEAL","OPEN_POS","candle")
df=(df.filter((F.floor(F.col("MOMENT")/1000000000)>=candle_date_from) &
              ((F.floor(F.col("MOMENT")/1000000000)<=candle_date_to)) &
              ((F.col("MOMENT")%10000000000)>=candle_time_from*100000) &
              ((F.col("MOMENT")%10000000000)<=candle_time_to*100000)))
a=(df.groupBy(F.col("MOMENT"),F.col("#SYMBOL"),F.col("candle"))
    .agg(F.min(df.PRICE_DEAL).alias("LOW"),
    F.max(df.PRICE_DEAL).alias("HIGH"),F.first(F.col("OPEN_POS")).alias("OPEN"),F.last(F.col("OPEN_POS")).alias("CLOSE")))

ans=a.select("#SYMBOL","MOMENT","OPEN","HIGH","LOW","CLOSE").orderBy(F.col("MOMENT"))
ans=ans.withColumn('HIGH',F.round(ans.HIGH,1))
ans=ans.withColumn('LOW',F.round(ans.LOW,1))
ans.show()
(ans.write.option("header",False)
        .partitionBy("#SYMBOL")
        .mode("overwrite")
        .csv("/result/candles"))